# 02.5 Transfer Learning / 迁移学习

迁移学习 / transfer learning 的核心思想是：  
The core idea of transfer learning is:

- 不从零开始训练全部知识 / do not learn everything from scratch
- 复用已有模型的特征提取能力 / reuse the feature-extraction ability of an existing model

重点概念 / Key concepts:

- 预训练 / pretraining
- 主干网络 / backbone
- 分类头 / classification head
- 冻结参数 / freeze parameters
- 解冻参数 / unfreeze parameters
- 微调 / fine-tuning

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 理解迁移学习的基本流程 / Understand the basic transfer-learning workflow.
2. 分清 backbone 和 head / Distinguish the backbone from the head.
3. 冻结和解冻参数 / Freeze and unfreeze parameters.
4. 替换分类头 / Replace a classification head.
5. 看懂 head-only training 和 fine-tuning 的区别 / Understand the difference between head-only training and fine-tuning.
6. 在离线环境下跑通一个最小迁移学习流程 / Run a minimal transfer-learning workflow in an offline environment.

## 离线说明 / Offline Note

为了保证这份 notebook 在当前离线环境里一定可运行，这里使用：  
To guarantee this notebook runs in the current offline environment, we use:

- `models.resnet18(weights=None)`

这意味着它不会真正利用预训练权重优势，但可以完整演示迁移学习流程。  
This means it will not benefit from pretrained weights, but it can still demonstrate the full transfer-learning workflow.

如果你未来在有缓存或联网的环境中，可改成：  
If you later work in an environment with cached weights or internet access, you can change it to:

- `models.resnet18(weights=models.ResNet18_Weights.DEFAULT)`

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

## 1. 准备一个可喂给 ResNet 的数据集
## Prepare a Dataset That Can Be Fed into ResNet

`digits` 是灰度图 `(1, 8, 8)`，而标准 `ResNet` 通常吃 RGB 图 `(3, H, W)`。  
`digits` is grayscale `(1, 8, 8)`, while a standard `ResNet` usually expects RGB images `(3, H, W)`.

所以这里做三步变换 / So we do three transformations:

1. 缩放到 `0~1` / scale to `0~1`
2. 灰度复制成 3 通道 / repeat grayscale into 3 channels
3. resize 到更适合 `ResNet` 的尺寸 / resize to a size more suitable for `ResNet`

In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

X_train_full, X_val, y_train_full, y_val = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

# 为了让 notebook 跑得更快，这里取一个较小子集做演示。
# To keep the notebook fast, we use a smaller subset for demonstration.
X_train = X_train_full[:256]
y_train = y_train_full[:256]
X_val_small = X_val[:96]
y_val_small = y_val[:96]

print("train subset shape =", X_train.shape)
print("val subset shape =", X_val_small.shape)

In [ ]:
transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Resize((64, 64), antialias=True),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class DigitsRGBDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(self.labels[index]), dtype=torch.long)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


train_ds = DigitsRGBDataset(X_train, y_train, transform=transform)
val_ds = DigitsRGBDataset(X_val_small, y_val_small, transform=transform)

xb, yb = train_ds[0]
print("one image shape =", xb.shape)
print("one label =", yb)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

xb, yb = next(iter(train_loader))
print("batch image shape =", xb.shape)
print("batch label shape =", yb.shape)

## 2. 构建 backbone + head
## Build the Backbone + Head

这里使用 `resnet18` 作为 backbone。  
We use `resnet18` here as the backbone.

分类头 / classification head 就是最后的 `fc` 层。  
The classification head is the final `fc` layer.

In [ ]:
model = models.resnet18(weights=None)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)

print(model.fc)
print("fc in_features =", in_features)

## 3. 冻结 backbone，只训练 head
## Freeze the Backbone and Train Only the Head

所谓 head-only training，就是：  
Head-only training means:

- 主干网络 / backbone 不更新
- 只训练最后分类头 / only the final classification head is updated

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print("trainable params =", trainable_params)
print("all params =", all_params)

## 4. 训练辅助函数 / Training Helper Functions

为了让 notebook 更清晰，先把训练逻辑封装起来。  
To keep the notebook clean, we wrap the training logic into helper functions first.

In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## 5. Head-only Training / 只训练分类头

因为 backbone 被冻结，优化器只接收 `requires_grad=True` 的参数。  
Because the backbone is frozen, the optimizer only receives parameters with `requires_grad=True`.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)

head_history = []
for epoch in range(1, 3):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    head_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "phase": "head_only",
    })
    print(f"head_only epoch={epoch} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

## 6. 解冻最后一部分并微调
## Unfreeze the Last Part and Fine-Tune

接下来只解冻 `layer4` 和 `fc`，让模型后部一起更新。  
Next we unfreeze only `layer4` and `fc`, so the later part of the model can update together.

这就是一个常见的部分微调 / partial fine-tuning 方案。  
This is a common partial fine-tuning setup.

In [ ]:
for param in model.layer4.parameters():
    param.requires_grad = True

trainable_params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable params after unfreezing layer4 =", trainable_params_after)

In [ ]:
finetune_optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

finetune_history = []
for epoch in range(1, 3):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=finetune_optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    finetune_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "phase": "partial_finetune",
    })
    print(f"partial_finetune epoch={epoch} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

## 7. 对比 head-only 和 partial fine-tuning
## Compare Head-only and Partial Fine-Tuning

这里的重点不是数字一定谁更高，而是理解两个阶段的训练方式不同。  
The key here is not that one result must always be better, but that the two stages train in different ways.

In [ ]:
history_df = torch.tensor([])  # placeholder to keep notebook structure simple
import pandas as pd

comparison = pd.DataFrame(head_history + finetune_history)
print(comparison)

## 8. 迁移学习的实践提醒 / Practical Notes on Transfer Learning

真正项目中，迁移学习通常更适合：  
In real projects, transfer learning is usually more suitable when:

- 数据量较小 / the dataset is relatively small
- 任务和已有视觉任务接近 / the new task is reasonably close to existing vision tasks
- 有可用预训练权重 / pretrained weights are available

而这份 notebook 的目标是让你先把“流程”彻底跑通。  
The goal of this notebook is to make sure you first understand the workflow end to end.

In [ ]:
# 练习 1 / Exercise 1
# 用一句话解释为什么 head-only training 时要冻结 backbone。
# In one sentence, explain why we freeze the backbone during head-only training.

参考回答 / Reference answer:

因为 head-only training 的目标是先只让新的分类头适配当前任务，而不立刻改动整个主干网络。  
Because the goal of head-only training is to let the new classification head adapt to the current task first, without immediately changing the whole backbone.

In [ ]:
# 练习 2 / Exercise 2
# 如果你想让更多层参与 fine-tuning，可以从当前 notebook 的哪里开始改？
# If you want more layers to participate in fine-tuning, where would you start modifying this notebook?

参考回答 / Reference answer:

可以从解冻更多 backbone 层开始，比如把 `layer3` 甚至更早的层也设为 `requires_grad=True`。  
You can start by unfreezing more backbone layers, such as setting `layer3` or even earlier layers to `requires_grad=True`.

## 9. 小结 / Summary

这一节最重要的是理解迁移学习的组织方式，而不只是会调用一个模型名字。  
The most important outcome of this notebook is understanding how transfer learning is organized, not just knowing the name of a model.

你现在应该能回答 / You should now be able to answer:

1. backbone 和 head 分别是什么？ / What are the backbone and the head?
2. 为什么有时要先 freeze，再 unfreeze？ / Why do we sometimes freeze first and unfreeze later?
3. head-only training 和 partial fine-tuning 的区别是什么？ / What is the difference between head-only training and partial fine-tuning?
4. 为什么这份 notebook 虽然离线可跑，但不代表真正获得了预训练优势？ / Why does this notebook run offline but still not fully benefit from pretraining?

下一步建议 / Suggested next step:

- 进入 Phase 2 小项目 notebook，用实验视角比较不同图像模型 / Move to the Phase 2 project notebook and compare different image models through experiments.